# 02 DEM metadata and EDA

Use verified assets in the same working tree. Validate the registered pixel centre and NASA ROI, summarize windowed statistics and inspect terrain previews.

In [ ]:
from pathlib import Path
import os, sys
root=Path(os.environ.get("LUNARSIGHT_ROOT", Path.cwd())).resolve()
if not (root/"src/pipeline.py").exists() and (root.parent/"src/pipeline.py").exists():
    root=root.parent
assert (root/"src/pipeline.py").exists(), "Set LUNARSIGHT_ROOT to the extracted lunarsight-data folder."
sys.path.insert(0,str(root/"src"))
for name in ("reports","manifests","processed","figures"): (root/name).mkdir(exist_ok=True)

In [ ]:
from acquire import load, inventory, download
from terrain import site_geojson
from pipeline import stage_dem
from IPython.display import Image, display
import pandas as pd
old=load(root/"manifests/source_manifest.json")
records={x["id"]:x for x in old["records"]}
for rec in records.values():
    if rec["status"] in {"PASS","VERIFIED_BYTES"}:
        download(root,rec,offline=True)
for key in ["dem80","dem240","Site01_count","Site01_error"]:
    stage_dem(root,key,records,"Site01" if key.startswith("Site01") else None)
site=stage_dem(root,"Site01_dem",records,"Site01")
site_geojson(root/"processed/site_catalog.geojson",[site])
inventory(root,list(records.values()),old.get("horizons",[]))
display(pd.DataFrame([dict(id=k,**v["stats"]) for k,v in records.items() if "stats" in v]))
display(Image(filename=str(root/"figures/Site01_dem_eda.png")))